# Fetal Head Segmentation – Data Preparation

In [ ]:
import json
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import torch
import torch.nn.functional as F
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torchvision import tv_tensors
from torchvision.io import read_image
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.dataset import HeadSegmentationDataset
from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    create_ellipse_tensor,
    read_image_tensor,
    save_image_tensor,
    show_numpy_images,
    show_pytorch_images,
)

data_dir = root / "data"
dataset_name = "FETAL_HEAD_SEGMENTATION"
dataset_dir = data_dir / dataset_name

# Prepare FETAL_HEAD_SEGMENTATION Dataset

Build the unified head-segmentation dataset from multiple annotation sources, assemble a single `data.csv`, and split it into patient-disjoint train / val / test subsets.

## Prepare Segmentation Masks

Normalise raw segmentation masks from three annotation sources (TC/TV/Thalamic, HC18, and Fix ellipses) into a common format: single-channel binary head masks stored as PNG files.

### TC / TV / Thalamic and HC18 Masks

Iterate over each annotation folder, convert multi-channel RGB masks to a unified red-channel head mask, and collect any image that produced no usable mask into `unselected_images`.

In [ ]:
def prepare_segmentation_masks(folder):
    data_path = dataset_dir / folder
    images_path = data_path / "Images"
    segmentations_path = data_path / "SegmentationClass"

    images = list(images_path.iterdir())
    for image_path in tqdm(images, desc="Prepare images"):

        segmentation_path = segmentations_path / image_path.name
        segmentation = np.array(Image.open(segmentation_path))

        red_color = [255, 0, 0]
        green_color = [0, 255, 0]
        blue_color = [0, 0, 255]

        if np.any((segmentation == red_color).all(axis=-1)):

            mask = (
                (segmentation == red_color).all(axis=-1)
                | (segmentation == green_color).all(axis=-1)
                | (segmentation == blue_color).all(axis=-1)
            )
            segmentation[mask] = red_color

            segmentation = Image.fromarray(segmentation)
            segmentation_path = data_path / "Segmentations" / segmentation_path.name
            # segmentation.save(segmentation_path)
        else:
            unselected_images.append((images_path / image_path.name, segmentations_path / image_path.name))


unselected_images = []
prepare_segmentation_masks(folder="Trans-cerebellum")
prepare_segmentation_masks(folder="Trans-thalamic")
prepare_segmentation_masks(folder="Trans-ventricular")
prepare_segmentation_masks(folder="HC18")

### Inspect Unselected Images

Display image / mask pairs that were collected in `unselected_images` during mask normalisation — frames for which no valid head mask could be produced and that will be excluded from the dataset.

In [ ]:
images = []
for image_path, segmentation_path in unselected_images:
    if image_path.exists():
        image = read_image(image_path)
        image = TF.to_grayscale(image)
        images.append((image, f"{image_path.stem}"))

    if segmentation_path.exists():
        image = read_image(segmentation_path)
        images.append((image))

if len(images) > 0:
    show_pytorch_images(images).show()

### Fill Missing FETAL_PLANES Masks

Some FETAL_PLANES images have no pixel-level annotation. For these, ellipse parameters stored in `FETAL_PLANES_SEGMENTATION_FIX.json` are used to generate synthetic binary masks.

#### Read Ellipse Annotations

Parse `FETAL_PLANES_SEGMENTATION_FIX.json` to extract the list of per-image ellipse parameters (centre, axes, angle).

In [ ]:
fix_path = dataset_dir / "FETAL_PLANES_SEGMENTATION_FIX.json"
with open(fix_path, "r") as fix_file:
    fix_data = json.load(fix_file)

extracted_data = []
for image_info in fix_data.values():
    filename = image_info.get("filename")
    regions = image_info.get("regions", [])

    if not regions:
        print(filename)

    for region in regions:
        shape_attributes = region.get("shape_attributes", {})

        if shape_attributes.get("name") == "ellipse":
            ellipse_data = {
                "filename": filename,
                "cx": shape_attributes.get("cx"),
                "cy": shape_attributes.get("cy"),
                "rx": shape_attributes.get("rx"),
                "ry": shape_attributes.get("ry"),
                "theta": shape_attributes.get("theta"),
            }
            extracted_data.append(ellipse_data)

len(pd.DataFrame(extracted_data))

#### Generate Ellipse Masks

For each entry in `extracted_data`, look up the corresponding image in `data_fix.csv`, draw the ellipse onto a blank canvas, and save the result as a binary PNG under `Fix/Segmentation/`.

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")

for ellipse_data in tqdm(extracted_data):
    image_name = ellipse_data["filename"]
    cx = ellipse_data["cx"]
    cy = ellipse_data["cy"]
    rx = ellipse_data["rx"]
    ry = ellipse_data["ry"]
    theta = ellipse_data["theta"]

    if (dataset_dir / "Fix" / "Segmentations" / image_name).exists():
        continue

    image_path = data_dir / "FETAL_PLANES" / "Images" / image_name
    image = read_image_tensor(image_path)

    mask = create_ellipse_tensor(image.shape[1], image.shape[2], cx, cy, rx, ry, theta)
    mask = mask.unsqueeze(0)
    mask = mask.repeat(3, 1, 1)
    mask = mask * 255
    mask = torch.clamp(mask, min=0, max=255)
    mask = mask.to(torch.uint8)

    image_path = dataset_dir / "Fix" / "Images" / image_name
    save_image_tensor(image, image_path)

    mask_path = dataset_dir / "Fix" / "Segmentations" / image_name
    save_image_tensor(mask, mask_path)

### JNU-IFM Masks (skip – low image quality)

Full pipeline for copying and remapping JNU-IFM ultrasound images and masks into the dataset format.
This source was evaluated but excluded due to low image quality.

In [ ]:
def prepare_segmentation_masks():
    data_path = data_dir / "JNU-IFM" / "us_data"

    data = []
    videos = list(data_path.iterdir())
    for video_id, video_path in enumerate(tqdm(videos, desc="Prepare videos")):
        images_path = video_path / "image"
        segmentations_path = video_path / "mask_enhance"

        images = list(images_path.iterdir())
        for image_path in tqdm(images, desc="Prepare images", leave=False):

            image = Image.open(image_path)
            image_path = dataset_dir / "JNU-IFM" / "Images" / image_path.name
            image.save(image_path)

            segmentation_path = segmentations_path / f"{image_path.stem}_mask.png"
            segmentation = np.array(Image.open(segmentation_path))

            black_color = [0, 0, 0]
            red_color = [255, 0, 0]
            green_color = [0, 255, 0]
            blue_color = [0, 0, 255]

            brain_plane = 1 if np.any((segmentation == green_color).all(axis=-1)) else 0

            segmentation[(segmentation == red_color).all(axis=-1)] = black_color
            segmentation[(segmentation == green_color).all(axis=-1)] = red_color

            segmentation = Image.fromarray(segmentation)
            segmentation_path = dataset_dir / "JNU-IFM" / "Segmentations" / image_path.name
            segmentation.save(segmentation_path)

            data.append(
                {
                    "Image_name": image_path.stem,
                    "Patient_num": video_id,
                    "Frame_num": int(image_path.stem.replace(video_path.name + "_", "")),
                    "Brain_plane": brain_plane,
                    "Ultrasound_path": str(image_path.relative_to(dataset_dir)),
                    "Segmentation_path": str(segmentation_path.relative_to(dataset_dir)),
                }
            )

    return data


data = prepare_segmentation_masks()
data_df = pd.DataFrame(data)
data_df = data_df.sort_values(["Patient_num", "Frame_num"])
data_df = data_df.reset_index(drop=True)
data_df.to_csv(dataset_dir / "JNU-IFM" / "data.csv", index=False)
data_df

### PSFHS Masks (skip)

Commented-out SimpleITK loader for the PSFHS dataset.
Not used in the final dataset.

In [ ]:
# import SimpleITK as sitk


# images_path = data_dir / "PSFHS" / "image_mha"
# segmentations_path = data_dir / "PSFHS" / "label_mha"

# images_ = []

# images = list(images_path.iterdir())
# for image_path in tqdm(images, desc="Prepare images", leave=False):
#     mask_path = segmentations_path / image_path.name

#     image_sitk = sitk.ReadImage(image_file)
#     mask_sitk = sitk.ReadImage(mask_path, sitk.sitkUInt8)

#     image_array = np.transpose(sitk.GetArrayFromImage(image_sitk), (1, 2, 0))
#     mask_array = sitk.GetArrayFromImage(mask_sitk)
#     images_.extend([image_array, mask_array])

#     if len(images_) >= 1000:
#         break

# show_numpy_images(images_[:64])

## Prepare CSV

Merge all image/mask sources into a single `data_df` dataframe, enrich it with patient metadata and split assignments from `FETAL_PLANES`, and write the final `data.csv`.

### Index Images with Masks

Scan the TC/TV/Thalamic, Fix, and HC18 directories and collect every (image, mask) path pair into `data_df`.
Each row records the image filename, mask path, and source brain-plane label.

In [ ]:
def get_segmentation_masks_data(folder, brain_plane):
    data_path = dataset_dir / folder
    images_path = data_path / "Images"
    segmentations_path = data_path / "Segmentations"

    images = list(images_path.iterdir())
    for image_path in tqdm(images, desc="Prepare images"):

        segmentation_path = segmentations_path / image_path.name

        if segmentation_path.exists():
            data.append(
                {
                    "Image_name": image_path.stem,
                    "Patient_num": None,
                    "Brain_plane": 1,
                    "Plane": brain_plane,
                    "Ultrasound_path": str(image_path.relative_to(dataset_dir)),
                    "Segmentation_path": str(segmentation_path.relative_to(dataset_dir)),
                }
            )
        # break


data = []
get_segmentation_masks_data(
    folder="Trans-cerebellum",
    brain_plane="Trans-cerebellum",
)
get_segmentation_masks_data(
    folder="Trans-thalamic",
    brain_plane="Trans-thalamic",
)
get_segmentation_masks_data(
    folder="Trans-ventricular",
    brain_plane="Trans-ventricular",
)
get_segmentation_masks_data(
    folder="Fix",
    brain_plane="Other",
)
get_segmentation_masks_data(
    folder="HC18",
    brain_plane="Other",
)

print(len(data))
data_df = pd.DataFrame(data)
data_df

### Load FETAL_PLANES Metadata

Load `FETAL_PLANES/data_fix.csv` to retrieve patient numbers, validity flags, and corrected brain-plane labels for images that originate from FETAL_PLANES.

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
fetal_planes

### Add 'Not A Brain' Images

Append rows for FETAL_PLANES images labelled `'Not A Brain'`.
These frames have no segmentation mask but are needed as negative examples for the binary head-detection head of the model.
Their `Subset` assignment is inherited from `data_fix.csv`.

In [ ]:
fetal_not_a_brain_planes = fetal_planes[fetal_planes["Brain_plane"] == "Not A Brain"]
for _, row in tqdm(fetal_not_a_brain_planes.iterrows(), total=len(fetal_not_a_brain_planes)):

    existing_index = data_df[data_df["Image_name"] == row["Image_name"]].index.to_list()
    if existing_index:
        index = existing_index[0]
    else:
        index = len(data_df)

    data_df.loc[index] = {
        "Image_name": row["Image_name"],
        "Brain_plane": 0,
        "Plane": "Not A Brain",
        "Ultrasound_path": f"FETAL_PLANES/Images/{row["Image_name"]}.png",
        "Segmentation_path": None,
    }


print(len(data_df))
data_df
len(data_df[data_df["Brain_plane"] == 1])

### Propagate FETAL_PLANES Metadata

For every row that originates from FETAL_PLANES, copy `Patient_num`, `Valid`, and the corrected `New_brain_plane` label from `data_fix.csv` into `data_df`.

In [ ]:
data_df["Patient_num"] = None
data_df["Valid"] = 1
new_brain_images = 0

for _, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    for index, _ in data_df[data_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Patient_num"] = str(row["Patient_num"])
        data_df.loc[index, "Valid"] = 1 if (pd.isna(row["Duplicate"]) and row["Valid"] == 1) else 0
        if isinstance(row["New_brain_plane"], str):
            data_df.loc[index, "Brain_plane"] = 1
            data_df.loc[index, "Segmentation_path"] = f"Fix/Segmentations/{data_df.Image_name[index]}.png"
            new_brain_images += 1

print(f"Images:           {len(data_df)}")
print(f"Brain images:     {len(data_df[data_df["Brain_plane"] == 1])}")
print(f"New brain images: {new_brain_images}")

### Exclude Mislabelled HC18 Brain Images

A curated list of images that are labelled as a brain plane but contain no detectable brain structure.
These are marked `Valid=0` and displayed for confirmation.

In [ ]:
images = []

remove_images = [
    # "009_HC",
    # "140_HC",
    # "637_HC",
    # "044_HC",
    # "096_HC",
    # "110_HC",
]
for image_name in remove_images:
    for index, row in data_df[data_df.Image_name == image_name].iterrows():
        data_df.loc[index, "Valid"] = 0

if images:
    show_numpy_images(images[:36])

### Add JNU-IFM Images (skip)

Merge rows from the JNU-IFM CSV into `data_df`.
Marked skip; the cell is kept for completeness but not executed.

In [ ]:
jnu_df = pd.read_csv(dataset_dir / "JNU-IFM" / "data.csv")

for _, row in tqdm(jnu_df.iterrows(), total=len(jnu_df)):

    existing_index = data_df[data_df["Image_name"] == row["Image_name"]].index.to_list()
    if existing_index:
        index = existing_index[0]
    else:
        index = len(data_df)

    data_df.loc[index] = {
        "Image_name": row["Image_name"],
        "Patient_num": str(row["Patient_num"] + 10000),
        "Brain_plane": row["Brain_plane"],
        "Plane": "Other" if row["Brain_plane"] == 1 else "Not A Brain",
        "Ultrasound_path": row["Ultrasound_path"],
        "Segmentation_path": row["Segmentation_path"] if row["Brain_plane"] == 1 else None,
    }


print(len(data_df))
print(len(data_df[data_df["Brain_plane"] == 1]))

### Assign Train and Subset

For images that appear in `FETAL_PLANES`, copy their `Train` and `Subset` values directly from `data_fix.csv` to keep splits consistent across experiments.
Images from HC18 and other external sources are assigned `Train=1` (training only).

In [ ]:
data_df["Train"] = 1
data_df["Subset"] = "train"

for _, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    for index, _ in data_df[data_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Train"] = row["Train"]
        data_df.loc[index, "Subset"] = row["Subset"]

### Statistics

Overall subset distribution, brain-plane class counts, and the number of valid brain images per split.

In [ ]:
data_df["Subset"].value_counts()

In [ ]:
data_df["Brain_plane"].value_counts()

In [ ]:
df = data_df[data_df["Brain_plane"] == 1]
df = df[df["Valid"] == 1]
df["Subset"].value_counts()

### Sort by Image Name

Sort `data_df` alphabetically by `Image_name` to ensure reproducible row ordering across runs.

In [ ]:
data_df = data_df.sort_values("Image_name")

### Save CSV

Write the merged and sorted dataframe to `FETAL_HEAD_SEGMENTATION/data.csv`.

In [ ]:
data_df.to_csv(dataset_dir / "data.csv", index=False)
data_df

## Split Data

Divide the dataset into patient-disjoint train / val / test subsets by scanning random seeds for well-balanced splits.

### Find Good Seeds

Scan candidate seeds for a 30 % test split (seed 2340 selected), then scan a second seed for a ~20 % validation split on the remaining training data.
Split quality is measured by class balance across subsets.

### Load Valid Rows

Reload the saved CSV and filter to rows where `Valid=1` and a `Patient_num` is available, ready for the seed-scanning split analysis.

In [ ]:
data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
data_df = data_df[data_df["Valid"] == 1]
data_df = data_df[data_df["Patient_num"].notna()]
data_df

In [ ]:
def group_split_label(
    dataset: pd.DataFrame, test_size: float, groups, random_state: int = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    splitter = GroupShuffleSplit(test_size=test_size, n_splits=1, random_state=random_state)
    split = splitter.split(dataset, groups=groups)
    train_idx, test_idx = next(split)
    return dataset.iloc[train_idx].reset_index(drop=True), dataset.iloc[test_idx].reset_index(drop=True)


def get_similarity(train, test, test_size):
    similarity = 0
    train_count = train.value_counts(sort=False).sort_index()
    test_count = test.value_counts(sort=False).sort_index()

    if train_count.index.tolist() != test_count.index.tolist():
        return -1

    for a, b in zip(train_count, test_count):
        similarity += (a * test_size - b * (1 - test_size)) ** 2
    return similarity**0.5


def plt_value_counts(ax, dataset, tile=None):
    counts = dataset.value_counts(sort=False).sort_index()
    counts.plot(kind="bar", ax=ax)
    if tile:
        ax.set_title(tile)


def plt_group_split(dataset: pd.DataFrame, test_size: float, random_states: list[int], top_states: int = None):
    splits = []
    for random_state in tqdm(random_states):
        tr, val = group_split_label(
            dataset,
            test_size=test_size,
            groups=dataset["Patient_num"],
            random_state=random_state,
        )

        similarity = get_similarity(tr.Brain_plane, val.Brain_plane, test_size)
        if similarity >= 0:
            splits.append((similarity, tr, val, random_state))

    splits.sort(key=lambda e: (e[0], e[3]))
    nrows = top_states if top_states else len(splits)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=2,
        sharex="all",
        squeeze=False,
        figsize=(20, 3 * nrows),
    )
    fig.suptitle(f"Test size {test_size}")
    for i, (similarity, tr, val, random_state) in enumerate(splits[:nrows]):
        plt_value_counts(axes[i, 0], tr.Brain_plane, tile=f"Seed {random_state}")
        plt_value_counts(axes[i, 1], val.Brain_plane, tile=f"Similarity {similarity}")

    plt.show()
    print([random_state for (similarity, tr, val, random_state) in splits[:nrows]])


plt_group_split(
    data_df,
    test_size=0.3,
    random_states=list(range(10000)),
    top_states=10,
)  # [7312, 2481, 3290, 7276, 2340, 9292, 1871, 7795, 6620, 5542]

In [ ]:
data_train, data_test = group_split_label(
    data_df,
    test_size=0.3,
    groups=data_df["Patient_num"],
    random_state=2340,
)

plt_group_split(
    data_train,
    test_size=0.2,
    random_states=list(range(10000)),
    top_states=10,
)
# 7312 -> 782, 3773, 5264, 9396, 2895
# 2481 -> 1078, 7503, 1474, 3427, 5375
# 3290 -> 391, 5289, 6328, 1020, 3168
# 7276 -> 6904, 2286, 7276, 1691, 1157
# 2340 -> 4261, 1993, 336, 1574, 6101

### Apply Split

Apply the chosen seed pair to assign the final `Subset` values in the full CSV.
The resulting partition is printed below.

In [ ]:
import random


def random_seeds(data):
    random_row = random.choice(data)
    seed_1 = random_row[0]
    seed_2 = random.choice(random_row[1])
    return seed_1, seed_2


def split_dataset(df, seed_1, seed_2):
    train_df, test_df = group_split_label(
        df,
        test_size=0.4,
        groups=df["Patient_num"],
        random_state=seed_1,
    )
    train_df = train_df.reset_index(drop=True)

    train_df, val_df = group_split_label(
        train_df,
        test_size=0.2,
        groups=train_df["Patient_num"],
        random_state=seed_2,
    )
    return train_df, val_df, test_df


data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
data_df = data_df[data_df["Valid"] == 1]
data_df = data_df[data_df["Patient_num"].notna()]

seeds = [
    [7312, [782, 3773, 5264, 9396, 2895]],
    [2481, [1078, 7503, 1474, 3427, 5375]],
    [3290, [391, 5289, 6328, 1020, 3168]],
    [7276, [6904, 2286, 7276, 1691, 1157]],
    [2340, [4261, 1993, 336, 1574, 6101]],
]
seed_1, seed_2 = random_seeds(seeds)
train_df, val_df, test_df = split_dataset(data_df, seed_1, seed_2)

data_df = pd.read_csv(dataset_dir / "data.csv", dtype={"Patient_num": str})
data_df["Subset"] = "train"
for index, row in tqdm(data_df.iterrows(), total=len(data_df)):
    for _ in train_df[train_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Subset"] = "train"
    for _ in val_df[val_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Subset"] = "val"
    for _ in test_df[test_df["Image_name"] == row["Image_name"]].iterrows():
        data_df.loc[index, "Subset"] = "test"


df = data_df[data_df["Brain_plane"] == 1]
df = df[df["Valid"] == 1]
df["Subset"].value_counts()

### Split Results

Final partition sizes after applying the selected seeds:

```
train    2298
test     1450
val       339
```

In [ ]:
data_df.to_csv(dataset_dir / "data.csv", index=False)
data_df

# Dataset Inspection

Visually verify that `HeadSegmentationDataset` loads images and masks correctly after the CSV is finalised.

## Visualisation Helpers and Sample Inspection

Define three display helpers — `display_image`, `overlap_mask`, and `display_image_and_mask` — then use them to inspect samples at indices 2013, 2046, and 6552.
Each sample is shown as raw image, binary mask, and blended overlay.

In [ ]:
def display_image(img, ax, title):
    img = img.detach()
    img = TF.to_pil_image(img)
    img = TF.to_grayscale(img)
    ax.imshow(np.asarray(img), cmap="gray")
    ax.set_title(title)
    ax.axis("off")


def overlap_mask(image: torch.Tensor, mask: torch.Tensor, ax, title):
    image = image.clone()
    image = TF.to_grayscale(image)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    if image.max() <= 1.0:
        image = image * 255

    image = image.int()
    image[0] = image[0] + mask * 255 * 0.4
    image = torch.clamp(image, min=0, max=255)
    image = image.to(torch.uint8)

    image = TF.to_image(image)
    image = image.permute(1, 2, 0).numpy()

    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")


def display_image_and_mask(image, mask):
    fig, axes = plt.subplots(ncols=3, nrows=1, squeeze=False, figsize=(15, 4))
    mask = mask.float()
    display_image(image, axes[0, 0], "Image")
    display_image(mask, axes[0, 1], "Mask")
    overlap_mask(image, mask, axes[0, 2], "Overlayed Mask")

    plt.tight_layout()
    plt.show()


input_size = (192, 256)
transform = T.Compose(
    [
        T.Grayscale(),
        PadToAspectRatio(
            input_size,
            fill={tv_tensors.Image: 0, tv_tensors.Mask: 0, "others": None},
        ),
        Resize(
            input_size,
            interpolation={
                tv_tensors.Image: T.InterpolationMode.BILINEAR,
                tv_tensors.Mask: T.InterpolationMode.NEAREST_EXACT,
                "others": None,
            },
        ),
        T.ToDtype(dtype={tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.float32, "others": None}, scale=True),
    ]
)

dataset = HeadSegmentationDataset(
    data_dir=data_dir,
    subset="train",
    transform=None,
)

dataset_t = HeadSegmentationDataset(
    data_dir=data_dir,
    subset="train",
    transform=transform,
)

print(len(dataset))
image, mask, label = dataset[2013]
print("shape")
print(image.shape)
print(mask.shape)
print(label.shape)
print("values")
print(image.min(), image.max())
print(torch.unique(mask))
print(label)
print("")

image_t, mask_t, label_t = dataset_t[2046]
print("shape")
print(image_t.shape)
print(mask_t.shape)
print(label_t.shape)
print("values")
print(image_t.min(), image_t.max())
print(torch.unique(mask_t))
print(label_t)
print("types")
print(mask_t.dtype)
print(label_t.dtype)
print("")

image_t2, mask_t2, label_t2 = dataset_t[6552]
print("shape")
print(image_t2.shape)
print(mask_t2.shape)
print(label_t2.shape)
print("values")
print(image_t2.min(), image_t2.max())
print(torch.unique(mask_t2))
print(label_t2)
print("types")
print(mask_t2.dtype)
print(label_t2.dtype)


display_image_and_mask(image, mask)
display_image_and_mask(image_t, mask_t)
display_image_and_mask(image_t2, mask_t2)
# show_pytorch_images([image, mask, image_t, mask_t]).show()